# FlyBrain Phase 2 — connectome ingestion on Google Colab (free tier)

Walks through the full pipeline for a **locally provided** connectome file:

1. install the package
2. inspect the file's schema
3. validate the column mapping
4. import a budgeted subset
5. build a **sparse** graph
6. print statistics
7. visualise a **small extracted subgraph**

Two rules this notebook follows:

- **FlyBrain downloads nothing.** You obtain the dataset yourself from the
  official provider and either leave it on disk or upload it here. There is no
  download URL in this repository, and none should be added to one.
- **Never densify or NetworkX-ify the full connectome.** Only a small
  extracted subgraph is ever turned into a `networkx` object or plotted.

> Set `INPUT_PATH` below. Without a real file, the cells fall back to the test
> fixture in `tests/fixtures/`, which is **NOT real biological data**.


In [ ]:
# --- 0. Runtime check -------------------------------------------------------
# Colab free tier: Runtime > Change runtime type > T4 GPU. Nothing below needs a
# GPU; ingestion is CPU and memory bound, not compute bound.
import sys

print("python", sys.version.split()[0])
try:
    import google.colab  # noqa: F401

    print("running in Colab")
except ImportError:
    print("not in Colab; running locally")


In [ ]:
# --- 1. Install -------------------------------------------------------------
!pip install -q -e .
# Parquet input needs an engine. Free and open source; skip for CSV.
# !pip install -q pyarrow
print("installed")


## Provide the file

**Option A — a file already on disk** (e.g. a large export you kept locally):
set `INPUT_PATH` to that path.

**Option B — upload it** (only sensible for a small export; Colab's disk is
temporary and the file is gone when the runtime ends):


In [ ]:
from pathlib import Path

INPUT_PATH = "tests/fixtures/sample_edges.csv"  # <- replace with your own file
# INPUT_PATH = "/content/data/raw/my_export.csv"

# from google.colab import files
# uploaded = files.upload()
# INPUT_PATH = next(iter(uploaded))

print("using:", INPUT_PATH, Path(INPUT_PATH).is_file())


## 2. Inspect the schema

Reads only the first few thousand rows, so this is cheap even on a multi-GB
file. If the source and target columns are reported as unresolved, the import
will stop rather than guess: use the `--source-column` / `--target-column`
flags shown at the end of the output.


In [ ]:
from flybrain.data.schema import inspect_schema

schema = inspect_schema(INPUT_PATH, nrows=10_000)
print(schema.render())


## 3. Import a budgeted subset

A node/edge budget does two things: it bounds memory, and it lets the reader
stop early instead of parsing the whole file. On a 2,000,000-edge file this
cell takes ~0.5 s and ~16 MB instead of ~17 s and ~690 MB.

Because the result is a subset, every statistic is labelled `SUBSET`. That label
is not decoration: it is the difference between a measurement and a sample.


In [ ]:
from flybrain.data.importer import ImportOptions, import_connectome
from flybrain.data.provenance import DataSource, Provenance
from flybrain.data.validation import ValidationConfig

# Provenance is declared by you and never guessed. Fill in what the source
# actually states; leave the rest as None and the report will say so.
provenance = Provenance(
    source_name="replace-me",              # who published the data
    dataset_name="replace-me",             # this specific export
    source=DataSource.EXTERNAL,
    dataset_version=None,                   # only if the source states one
    source_url=None,                        # the page you downloaded from
    license=None,                           # e.g. "CC-BY-4.0" if that is the licence
    citation=None,
    notes="Replace this cell with your own provenance before reporting anything.",
)

options = ImportOptions(
    validation=ValidationConfig(
        max_nodes=5_000,
        max_edges=50_000,
        on_limit="subsample",   # "error" refuses instead of sampling
        seed=0,                  # reproducible subset
    ),
    chunksize=25_000,
    keep_extra_columns=False,   # lower memory; drop unmapped source columns
)

result = import_connectome(INPUT_PATH, provenance=provenance, options=options)
print(f"nodes={result.num_nodes:,}  edges={result.num_edges:,}  scope={result.scope}")


## 4. Dataset report

The report states its own provenance and whether it describes the complete
dataset or a subset. It also states that any neural activity later derived from
this graph would be *simulated*, not recorded.


In [ ]:
from flybrain.data.report import build_connectome_report

print(build_connectome_report(result.connectome, result.validation).render())


## 5. Sparse graph

The graph is stored as a `scipy` CSR matrix. For the 120k-neuron, 2M-edge scale
check used during development, CSR cost 16 MB where a dense `N x N` float32
matrix would have needed 58 GB.


In [ ]:
from flybrain.brain.graph import BrainGraph

graph = BrainGraph.from_connectome(result.connectome)
print(graph.summary())
print("is_biological:", graph.is_biological)
print(graph.provenance_warning() or "[Biological connectome data] topology from " + result.provenance.source_name)

n = graph.num_nodes
csr_bytes = graph.weights.data.nbytes + graph.weights.indices.nbytes + graph.weights.indptr.nbytes
print(f"\nsparse storage : {csr_bytes / 1e6:.1f} MB")
print(f"dense would be : {n * n * 4 / 1e9:.1f} GB  <- never do this")


## 6. Visualise a SMALL extracted subgraph

Take the most connected few hundred nodes, extract the induced subgraph, and
plot that. `BrainGraph.to_networkx()` refuses graphs above its node limit, so this
cannot accidentally materialise the full connectome.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

PLOT_NODES = 300
sub = graph.subgraph(graph.top_nodes(PLOT_NODES))
print(f"subgraph: {sub.num_nodes} nodes, {sub.num_edges} edges, sparse={sub.is_sparse}")

small = sub.to_networkx(max_nodes=PLOT_NODES)
pos = nx.spring_layout(small, seed=0)

fig, ax = plt.subplots(figsize=(9, 9))
ax.set_facecolor("#0b1020")
nx.draw_networkx_edges(small, pos, ax=ax, alpha=0.2, edge_color="#7f9cf5", width=0.5)
nx.draw_networkx_nodes(
    small, pos, ax=ax, node_color="#2ec4b6", node_size=40, edgecolors="#0b1020", linewidths=0.5
)
label = "Biological connectome data" if graph.is_biological else (
    "TEST FIXTURE - NOT REAL BIOLOGICAL DATA" if result.provenance.is_test_fixture else "Synthetic / simulated"
)
ax.set_title(
    f"{label}\n"
    f"source={result.provenance.source_name}  scope={result.scope}  "
    f"{sub.num_nodes} of {graph.num_nodes} nodes",
    fontsize=9, color="#c8d2f0",
)
ax.set_axis_off()
fig.tight_layout()
plt.show()


## 7. Save the processed subset

Writes the canonical edge table, a neuron table if one was supplied, and a
provenance sidecar. Commit none of it: `data/raw/` and `data/processed/` are
git-ignored.


In [ ]:
result = import_connectome(
    INPUT_PATH,
    provenance=provenance,
    options=options,
    output_dir="data/processed",
)
print("edges:", result.output_path)
print("provenance:", result.output_path.with_suffix("").with_suffix(".source.json"))

!ls -la data/processed/ 2>/dev/null || true


## What this notebook deliberately does not do

- No RL, no policy training.
- No visual circuit extraction.
- No neuron firing visualisation — activity in FlyBrain is simulated, and that
  belongs to a later phase with its own honesty requirements.
- No dashboard.

A connectome is **data**: a published measurement of which neurons connect to
which. It contains no weights, no dynamics and no learning rules. Anything this
project later computes over that structure is a simulation and must be reported
as one.
